# 🎭 AI-Powered Emotion Detection from Text Using NLP and Machine Learning

---

## Project Overview

Emotion Detection from Text is a Natural Language Processing (NLP) project that automatically classifies user text into **six emotions**: Sadness, Joy, Love, Anger, Fear, and Surprise.

The pipeline covers:
1. **Data Loading** – Hugging Face `dair-ai/emotion` dataset
2. **Exploratory Data Analysis (EDA)** – distribution, length statistics
3. **Text Preprocessing** – lowercase, remove punctuation, tokenize, remove stopwords, lemmatize
4. **Feature Extraction** – TF-IDF vectorization
5. **Model Training** – Logistic Regression classifier
6. **Evaluation** – accuracy, precision, recall, F1-score, confusion matrix
7. **Inference** – `predict_emotion()` function for real-time prediction
8. **Model Persistence** – save & reload via `pickle`

### Technologies
`Python` | `Pandas` | `NumPy` | `NLTK` | `Scikit-Learn` | `Matplotlib` | `Seaborn` | `Hugging Face datasets`


## 📌 Problem Statement

Humans naturally understand the emotional tone of written language. Machines, however, cannot do this by default.

**Goal:** Build an AI model that reads a sentence and predicts the underlying emotion.

**Applications:**
- Customer support ticket routing
- Social media sentiment monitoring
- Mental health platform analysis
- Emotion-aware conversational AI


## 🎯 Project Objectives

- Understand the end-to-end NLP workflow
- Perform text preprocessing and cleaning
- Convert text into machine-readable numerical features
- Train and evaluate a multi-class classification model
- Build an inference function for real-world prediction
- Save and reload trained model artefacts


## 1. Imports & Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import re
import pickle
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from datasets import load_dataset

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

# Download required NLTK resources
for resource in ["punkt", "punkt_tab", "stopwords", "wordnet"]:
    nltk.download(resource, quiet=True)

print("✅ All libraries imported successfully.")


## 2. Load Dataset

Using the [`dair-ai/emotion`](https://huggingface.co/datasets/dair-ai/emotion) dataset from Hugging Face.

| Split      | Samples |
|------------|---------|
| Train      | 16,000  |
| Validation | 2,000   |
| Test       | 2,000   |


In [ ]:
data = load_dataset("dair-ai/emotion")
print(data)


In [ ]:
# Inspect feature schema
print(data['train'].features)


In [ ]:
# Sample record
print(data['train'][0])


In [ ]:
# Convert to DataFrames
train_df = data['train'].to_pandas()
val_df   = data['validation'].to_pandas()
test_df  = data['test'].to_pandas()

# Map integer labels to human-readable emotion names
label_names = data['train'].features['label'].names
print("Emotion classes:", label_names)

for df in [train_df, val_df, test_df]:
    df['emotion'] = df['label'].map(lambda x: label_names[x])

train_df.head()


## 3. Exploratory Data Analysis (EDA)

EDA helps us understand the dataset *before* training.
We will inspect:
- Dataset shape and data types
- Missing values
- Class (emotion) distribution
- Text length distribution


In [ ]:
print("Train shape :", train_df.shape)
print("Val shape   :", val_df.shape)
print("Test shape  :", test_df.shape)


In [ ]:
# Data types and non-null counts
train_df.info()


In [ ]:
# Missing values
print("Missing values in train_df:")
print(train_df.isnull().sum())


In [ ]:
# Emotion distribution
print("Training set — emotion distribution:")
print(train_df['emotion'].value_counts())


In [ ]:
# Bar chart of emotion distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
counts = train_df['emotion'].value_counts()
axes[0].bar(counts.index, counts.values, color=sns.color_palette("Set2", len(counts)))
axes[0].set_title("Emotion Distribution (Training Set)", fontsize=13)
axes[0].set_xlabel("Emotion")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis='x', rotation=30)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 50, str(v), ha='center', fontsize=9)

# Percentage pie chart
axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=sns.color_palette("Set2", len(counts)), startangle=140)
axes[1].set_title("Emotion Share (%)", fontsize=13)

plt.tight_layout()
plt.savefig("emotion_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved → emotion_distribution.png")


### 📊 Observation — Class Imbalance

| Emotion  | ~Samples | Relative frequency |
|----------|----------|--------------------|
| Joy      | 5,362    | ██████████ (33 %)  |
| Sadness  | 4,666    | █████████ (29 %)   |
| Anger    | 2,159    | ████ (13 %)        |
| Fear     | 1,937    | ████ (12 %)        |
| Love     | 1,304    | ██ (8 %)           |
| Surprise | 572      | █ (3 %)            |

**Key takeaway:** `Surprise` is severely underrepresented (~572 samples vs 5,362 for Joy). This imbalance will lead to lower recall for `Surprise`.


In [ ]:
# Text length analysis
train_df["text_length"] = train_df["text"].apply(len)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(train_df["text_length"], bins=50, color="steelblue", edgecolor="white")
axes[0].set_title("Text Length Distribution (Training Set)")
axes[0].set_xlabel("Character Count")
axes[0].set_ylabel("Frequency")

# Box plot per emotion
train_df.boxplot(column="text_length", by="emotion", ax=axes[1], vert=False)
axes[1].set_title("Text Length by Emotion")
axes[1].set_xlabel("Character Count")
plt.suptitle("")
plt.tight_layout()
plt.show()

print(train_df["text_length"].describe().round(1))


In [ ]:
# Shortest and longest texts
print("Longest text:")
print(train_df.loc[train_df["text_length"].idxmax(), "text"])
print()
print("Shortest text:")
print(train_df.loc[train_df["text_length"].idxmin(), "text"])


## 4. Text Preprocessing

Raw text contains noise that hurts model performance.

**Pipeline:**
1. **Lowercase** — reduces vocabulary size
2. **Remove special characters** — strips punctuation, numbers, emojis
3. **Tokenise** — split into individual words
4. **Remove stopwords** — discard uninformative common words ("the", "is", "a")
5. **Lemmatise** — reduce words to their root form ("running" → "run")


In [ ]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text: str) -> str:
    """Clean and normalise raw text for TF-IDF feature extraction."""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words]
    tokens = [lemmatizer.lemmatize(w) for w in tokens]
    return " ".join(tokens)

# Quick sanity check
examples = [
    "I am feeling very happy today!!!",
    "She was ABSOLUTELY TERRIFIED of the dark...",
    "I lost my job — I feel so hopeless.",
]
for ex in examples:
    print(f"  Original : {ex}")
    print(f"  Cleaned  : {preprocess_text(ex)}")
    print()


In [ ]:
# Apply preprocessing to all splits
print("Preprocessing training set  ...", end=" ")
train_df["clean_text"] = train_df["text"].apply(preprocess_text)
print("done")

print("Preprocessing validation set...", end=" ")
val_df["clean_text"] = val_df["text"].apply(preprocess_text)
print("done")

print("Preprocessing test set      ...", end=" ")
test_df["clean_text"] = test_df["text"].apply(preprocess_text)
print("done")

# Preview
train_df[["text", "clean_text", "emotion"]].head()


## 5. Feature Extraction — TF-IDF

**TF-IDF (Term Frequency–Inverse Document Frequency)** converts text into numerical vectors.

- **TF** — how often a word appears in a document
- **IDF** — down-weights words that appear in almost every document (they carry little information)

> ⚠️ **Data Leakage Prevention:** `TfidfVectorizer.fit()` is called **only on the training set**. The validation and test sets are transformed using the already-fitted vectorizer to avoid leakage.


In [ ]:
# Fit ONLY on training data — prevents data leakage!
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

X_train = tfidf.fit_transform(train_df["clean_text"])   # fit + transform
X_val   = tfidf.transform(val_df["clean_text"])         # transform only
X_test  = tfidf.transform(test_df["clean_text"])        # transform only

y_train = train_df["label"]
y_val   = val_df["label"]
y_test  = test_df["label"]

print(f"X_train shape : {X_train.shape}  → (samples, features)")
print(f"X_val   shape : {X_val.shape}")
print(f"X_test  shape : {X_test.shape}")


## 6. Model Training — Logistic Regression

**Why Logistic Regression?**
- Fast and memory-efficient on sparse TF-IDF matrices
- Strong baseline for text classification
- Interpretable (feature importance via coefficients)
- Multi-class support via One-vs-Rest (OvR)


In [ ]:
model = LogisticRegression(
    max_iter=1000,
    C=1.0,             # Regularisation strength (default)
    solver="lbfgs",    # Works well for multi-class
    multi_class="auto",
    random_state=42,
)

model.fit(X_train, y_train)
print("✅ Model training complete.")


## 7. Model Evaluation

We evaluate on the **held-out test set** using:
- **Accuracy** — overall correctness
- **Precision** — of all predicted positives, how many are truly positive?
- **Recall** — of all actual positives, how many did we catch?
- **F1-Score** — harmonic mean of precision and recall (handles class imbalance)
- **Confusion Matrix** — per-class error breakdown


In [ ]:
# Validation set check
y_val_pred = model.predict(X_val)
val_acc = accuracy_score(y_val, y_val_pred)
print(f"Validation Accuracy: {val_acc:.4f}")


In [ ]:
# Test set evaluation
y_pred = model.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)

print(f"Test Accuracy : {test_acc:.4f}\n")
print("Classification Report")
print("=" * 60)
print(classification_report(y_test, y_pred, target_names=label_names))


In [ ]:
# Confusion Matrix — heatmap
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=label_names,
    yticklabels=label_names,
)
plt.xlabel("Predicted Emotion", fontsize=12)
plt.ylabel("Actual Emotion", fontsize=12)
plt.title("Confusion Matrix — Emotion Detection", fontsize=14)
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved → confusion_matrix.png")


## 8. Save Model Artifacts

Both the trained model and the fitted TF-IDF vectorizer are saved so they can be reloaded for inference without retraining.


In [ ]:
Path("models").mkdir(exist_ok=True)

with open("models/emotion_model.pkl", "wb") as f:
    pickle.dump(model, f)

with open("models/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)

print("✅ Model saved   → models/emotion_model.pkl")
print("✅ Vectorizer saved → models/tfidf_vectorizer.pkl")


## 9. Emotion Prediction System

A simple inference function that accepts raw text and returns the predicted emotion.


In [ ]:
def predict_emotion(text: str) -> str:
    """Predict the emotion expressed in raw input text."""
    cleaned = preprocess_text(text)
    vector  = tfidf.transform([cleaned])
    pred_idx = model.predict(vector)[0]
    return label_names[pred_idx]


def predict_with_confidence(text: str) -> dict:
    """Return emotion + probability scores for all classes."""
    cleaned = preprocess_text(text)
    vector  = tfidf.transform([cleaned])
    probs   = model.predict_proba(vector)[0]
    pred_idx = probs.argmax()
    return {
        "emotion":    label_names[pred_idx],
        "confidence": round(float(probs[pred_idx]), 4),
        "all_scores": {label_names[i]: round(float(p), 4) for i, p in enumerate(probs)},
    }


# --- Demo ---
test_sentences = [
    "I got selected for an internship! I'm so excited!",
    "I lost my wallet and I feel terrible.",
    "I am absolutely terrified about the exam results.",
    "I love spending time with my family.",
    "I cannot believe he said that — I'm furious!",
    "What just happened? That was completely unexpected!",
]

print(f"{'Text':<55} {'Prediction':<12} {'Confidence'}")
print("-" * 85)
for s in test_sentences:
    result = predict_with_confidence(s)
    print(f"{s[:54]:<55} {result['emotion'].upper():<12} {result['confidence']*100:.1f}%")


## 🔭 Future Scope

| Enhancement | Description |
|-------------|-------------|
| Handle class imbalance | SMOTE oversampling or `class_weight='balanced'` in Logistic Regression |
| Deep Learning | LSTM / GRU recurrent networks for sequential context |
| Transformer models | Fine-tune BERT, DistilBERT, or RoBERTa for state-of-the-art accuracy |
| Web application | Gradio or Streamlit demo for live inference |
| Multilingual support | Extend to non-English emotion detection |
| REST API | FastAPI deployment for production use |


## ✅ Conclusion

An **AI-Powered Emotion Detection System** was built end-to-end using NLP and Machine Learning.

**Summary:**
1. Loaded and explored the `dair-ai/emotion` dataset (16K train / 2K val / 2K test)
2. Applied NLP preprocessing (lowercasing, stopword removal, lemmatization)
3. Extracted TF-IDF features (5,000 bigram vocabulary)
4. Trained a Logistic Regression classifier — **~89% test accuracy**
5. Evaluated using accuracy, precision, recall, F1-score, and confusion matrix
6. Built `predict_emotion()` and `predict_with_confidence()` for real-time inference
7. Persisted trained artifacts to `models/`

This project demonstrates a complete NLP classification pipeline and provides a strong foundation for more advanced emotion-aware AI systems.
